# Paso 4 — Cargar la Fact Table (Silver + Gold → Gold)
**Archivo original:** `etl/ETL-FACT.py`

---

## ¿Qué hace este script?

Es el paso más importante del ETL: construye la **tabla de hechos** (`fact_aduana_item`)
cruzando el staging con todas las dimensiones mediante `LEFT JOIN`.

**El resultado:** cada fila del staging se convierte en una fila de la fact table,
pero en lugar de tener los datos descriptivos en texto, tiene solo los **IDs** de cada dimensión.

**¿Por qué LEFT JOIN y no INNER JOIN?**
Porque si usamos INNER JOIN, cualquier fila del staging que no tenga match
en alguna dimensión se perdería. Con LEFT JOIN, la fila se conserva y el ID queda NULL.


In [ ]:
import duckdb

DB_PATH = r"C:\Información\proyectos\aduana_bi\db\aduana.duckdb"
con = duckdb.connect(DB_PATH)

# Limpiar la fact table antes de cargar
con.execute("DELETE FROM dw.fact_aduana_item;")
print("fact_aduana_item limpiada. Cargando...")

---

## El INSERT principal: 14 JOINs en una sola consulta

Esta es la consulta central de todo el proyecto.
Se explica en secciones:

### Sección SELECT — ¿qué columnas se seleccionan?

| Columna | Fuente | Descripción |
|---------|--------|-------------|
| `id_fact` | ROW_NUMBER() | ID autoincremental generado |
| `despacho_cifrado`, `item` | `s` (staging) | Clave natural del negocio |
| `id_operacion` | `o` (dim_operacion) | FK hacia la dimensión |
| `id_pais_origen` | `po` (dim_pais) | dim_pais usada con alias 'po' |
| `id_pais_destino` | `pd` (dim_pais) | misma dim_pais con alias 'pd' |
| `fob_dolar`, `total`, ... | `s` (staging) | Métricas numéricas |

### Sección FROM + JOINs — ¿cómo se relacionan?

Cada dimensión se une con la condición que corresponde:
- **Texto simple:** `TRIM(s.campo) = TRIM(dim.campo)`
- **País (parseo):** `TRIM(SPLIT_PART(s.pais_origen, ' - ', 1)) = TRIM(po.codigo_pais)`
- **Producto (compuesto):** 6 condiciones AND con COALESCE
- **Fecha:** `s.oficializacion = fo.fecha`


In [ ]:
con.execute("""
INSERT INTO dw.fact_aduana_item (
    id_fact,
    despacho_cifrado, item,
    id_operacion, id_destinacion, id_regimen, id_aduana,
    id_pais_origen, id_pais_destino,
    id_producto,
    id_medio_transporte, id_canal, id_unidad_medida,
    id_acuerdo, id_marca,
    id_fecha_oficializacion, id_fecha_cancelacion,
    uso,
    cantidad_estadistica, kilo_neto, kilo_bruto,
    fob_dolar, flete_dolar, seguro_dolar,
    imponible_dolar, imponible_gs,
    ajuste_a_incluir, ajuste_a_deducir,
    derecho, isc, servicio, renta, iva, otros, total
)
SELECT
    -- ID surrogate: número de fila ordenado por despacho e ítem
    ROW_NUMBER() OVER (
        ORDER BY COALESCE(s.despacho_cifrado, ''), COALESCE(s.item, 0)
    ) AS id_fact,

    -- Clave natural del negocio
    s.despacho_cifrado,
    s.item,

    -- IDs de dimensiones (vienen del JOIN con cada tabla dim_xxx)
    o.id_operacion,
    d.id_destinacion,
    r.id_regimen,
    a.id_aduana,

    -- dim_pais se usa dos veces con alias distintos
    po.id_pais AS id_pais_origen,
    pd.id_pais AS id_pais_destino,

    p.id_producto,
    mt.id_medio_transporte,
    c.id_canal,
    um.id_unidad_medida,
    ac.id_acuerdo,
    m.id_marca,

    -- dim_fecha se usa dos veces con alias distintos
    fo.id_fecha AS id_fecha_oficializacion,
    fc.id_fecha AS id_fecha_cancelacion,

    -- Atributo descriptivo (no es FK)
    s.uso,

    -- Métricas numéricas (directo del staging)
    s.cantidad_estadistica,
    s.kilo_neto,
    s.kilo_bruto,
    s.fob_dolar,
    s.flete_dolar,
    s.seguro_dolar,
    s.imponible_dolar,
    s.imponible_gs,
    s.ajuste_a_incluir,
    s.ajuste_a_deducir,
    s.derecho,
    s.isc,
    s.servicio,
    s.renta,
    s.iva,
    s.otros,
    s.total

-- Tabla base: el staging completo
FROM dw.stg_aduana s

-- JOIN simple por texto (con TRIM para eliminar espacios)
LEFT JOIN dw.dim_operacion o
    ON TRIM(s.operacion) = TRIM(o.operacion)

LEFT JOIN dw.dim_destinacion d
    ON TRIM(s.destinacion) = TRIM(d.cod_destinacion)

LEFT JOIN dw.dim_regimen r
    ON TRIM(s.regimen) = TRIM(r.regimen)

LEFT JOIN dw.dim_aduana a
    ON TRIM(s.aduana) = TRIM(a.aduana)

-- JOIN por código de país (extraído del formato 'COD - DESCRIPCION')
LEFT JOIN dw.dim_pais po
    ON TRIM(SPLIT_PART(s.pais_origen, ' - ', 1)) = TRIM(po.codigo_pais)

LEFT JOIN dw.dim_pais pd
    ON TRIM(SPLIT_PART(s.pais_procedencia_destino, ' - ', 1)) = TRIM(pd.codigo_pais)

-- JOIN compuesto por 6 campos (la clave de producto es combinada)
LEFT JOIN dw.dim_producto p
    ON COALESCE(TRIM(s.posicion),      '') = COALESCE(TRIM(p.posicion_ncm),   '')
   AND COALESCE(TRIM(s.rubro),         '') = COALESCE(TRIM(p.rubro),           '')
   AND COALESCE(TRIM(s.desc_capitulo), '') = COALESCE(TRIM(p.desc_capitulo),  '')
   AND COALESCE(TRIM(s.desc_posicion), '') = COALESCE(TRIM(p.desc_posicion),  '')
   AND COALESCE(TRIM(s.desc_partida),  '') = COALESCE(TRIM(p.desc_partida),   '')
   AND COALESCE(TRIM(s.mercaderia),    '') = COALESCE(TRIM(p.mercaderia),      '')

-- JOINs simples restantes
LEFT JOIN dw.dim_medio_transporte mt
    ON TRIM(s.medio_transporte) = TRIM(mt.medio_transporte)

LEFT JOIN dw.dim_canal c
    ON TRIM(s.canal) = TRIM(c.canal)

LEFT JOIN dw.dim_unidad_medida um
    ON TRIM(s.unidad_medida_estadistica) = TRIM(um.unidad_medida)

LEFT JOIN dw.dim_acuerdo ac
    ON TRIM(s.acuerdo) = TRIM(ac.acuerdo)

LEFT JOIN dw.dim_marca m
    ON TRIM(s.marca_item) = TRIM(m.marca)

-- JOIN por fecha directa (tipo DATE)
LEFT JOIN dw.dim_fecha fo
    ON s.oficializacion = fo.fecha

LEFT JOIN dw.dim_fecha fc
    ON s.cancelacion = fc.fecha

-- Filtro: solo filas con clave natural completa
WHERE s.despacho_cifrado IS NOT NULL
  AND s.item IS NOT NULL;
""")

n = con.execute("SELECT COUNT(*) FROM dw.fact_aduana_item").fetchone()[0]
print(f"fact_aduana_item cargada: {n} filas")

---

## Verificación rápida del resultado


In [ ]:
# Ver una muestra de la fact table
muestra = con.execute("""
    SELECT
        id_fact,
        despacho_cifrado,
        item,
        id_operacion,
        id_pais_origen,
        id_producto,
        fob_dolar,
        total
    FROM dw.fact_aduana_item
    LIMIT 5
""").fetchdf()
print(muestra.to_string())

In [ ]:
# Ejemplo de consulta analítica: total FOB por operación
resultado = con.execute("""
    SELECT
        o.operacion,
        COUNT(*) AS cantidad_items,
        ROUND(SUM(f.fob_dolar), 2) AS total_fob_usd
    FROM dw.fact_aduana_item f
    JOIN dw.dim_operacion o ON f.id_operacion = o.id_operacion
    GROUP BY o.operacion
    ORDER BY total_fob_usd DESC
""").fetchdf()
print(resultado.to_string())

In [ ]:
con.close()
print("Fact table generada correctamente.")

---

## Resumen conceptual

```
stg_aduana (texto)         →    fact_aduana_item (IDs + métricas)
───────────────────────────────────────────────────────────────────
operacion = 'IMPORTACION'  →    id_operacion = 1
aduana = 'ASUNCION'        →    id_aduana = 2
pais_origen = 'BRA - ...'  →    id_pais_origen = 7
fob_dolar = 1500.00        →    fob_dolar = 1500.00  (sin cambio)
```

**Siguiente paso:** `05_Validaciones.ipynb`
